In [0]:
from datetime import date, timedelta
from pyspark.sql import functions as F, Window
from pyspark.sql.types import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
# Constants
SOURCE_TABLE_CONF = {
    "table": "customers_raw",
    "schema": "bronze",
}

TARGET_TABLE_CONF = {
    "table": "pyspark_scd2_customers",
    "schema": "silver",
    "primary_key": "customer_id"
}

In [0]:
# Configs
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())
today = date.today() - timedelta(days=1)

ENV = configs.get("env", "dev")
INITIAL_RUN = configs.get('initial_run', False)
START_DATE = configs.get("start_date") or today - timedelta(days=1)
END_DATE = configs.get("end_date") or today
CATALOG = f"sl_{ENV}"
CHECKPOINT_BASE = f"""/Volumes/{CATALOG}/{TARGET_TABLE_CONF.get("schema")}/checkpoints"""

print(ENV, INITIAL_RUN, START_DATE, END_DATE, CATALOG)

In [0]:
source_table = f"""{SOURCE_TABLE_CONF.get("schema")}.{SOURCE_TABLE_CONF.get("table")}"""
target_table = f"""{TARGET_TABLE_CONF.get("schema")}.{TARGET_TABLE_CONF.get("table")}"""
checkpoint_path = f"""{CHECKPOINT_BASE}/{TARGET_TABLE_CONF.get("table")}/"""
print(source_table, target_table, checkpoint_path)

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")

In [ ]:
if INITIAL_RUN: #this is not an ideal production pattern but more straight forward for this used case
    import shutil
    try:
        shutil.rmtree(checkpoint_path)
    except FileNotFoundError:
        print("no chekpoints found")

    empty_df = (spark.createDataFrame([], spark.read.table(source_table).schema)
            .withColumn("start_at", F.lit(None).cast(TimestampType()))
            .withColumn("end_at", F.lit(None).cast(TimestampType()))
            .withColumn("is_current", F.lit(None).cast(BooleanType()))
            .withColumn("_insert_ts", F.lit(None).cast(TimestampType()))
            .withColumn("_update_ts", F.lit(None).cast(TimestampType()))
            .withColumn("_is_delete", F.lit(None).cast(BooleanType()))
            .drop("last_updated", "_change_type", "_commit_timestamp", "_ingested_at", "_commit_version")
            .withColumn("_commit_version", F.lit(None).cast(LongType())) 
            )
    
    primary_key = TARGET_TABLE_CONF.get('primary_key')

    (empty_df
     .writeTo(target_table)
     .using("delta")
     .createOrReplace())
    
    spark.sql(f"ALTER TABLE {target_table} CLUSTER BY (customer_id, start_at, end_at)")
    spark.sql(f"ALTER TABLE {target_table} ALTER COLUMN {primary_key} SET NOT NULL")
    spark.sql(f"ALTER TABLE {target_table} ADD CONSTRAINT {primary_key}_pk PRIMARY KEY ({primary_key})")

In [0]:
df = (
    spark.readStream
    .option("startingVersion", 0)
    .table(source_table)
    .withColumn("_process_ts", F.current_timestamp())
    )

In [0]:
def merge_to_dim_customer(batch_df, batch_id):
    target = DeltaTable.forName(spark, target_table)

    # refine batch_df, and account for multiple records in same batch per customer_id
    window_spec = Window.partitionBy("customer_id").orderBy("_commit_version")
    refined_batch_df = (batch_df.filter(F.col("_change_type") != "update_preimage")
                        .withColumn("start_at", F.col("last_updated"))
                        .withColumn("next_start", F.lead(F.col("last_updated")).over(window_spec))
                        .withColumn("end_at", F.expr("timestampadd(MILLISECOND, -1, next_start)"))
                        .withColumn("is_current", F.col("next_start").isNull())
                        .withColumn("sequence_order", F.row_number().over(window_spec))
    )

    # close current rows that have changed matched on key column and commit version for idempotency
    (target.alias("t").merge(
      refined_batch_df.filter((F.col("_change_type") != "delete") & (F.col("sequence_order") == 1)).alias("s"),
      "s.customer_id = t.customer_id and t.is_current and s._commit_version > t._commit_version")
    .whenMatchedUpdate(set = {"end_at": F.expr("timestampadd(MILLISECOND, -1, s.start_at)"),
                              "is_current": F.lit(False),
                              "_update_ts": "s._process_ts"})
    .execute()
    ) 
    
    # insert new rows matched on key column and commit version for idempotency
    (target.alias("t").merge(
      refined_batch_df.filter(F.col("_change_type") != "delete").alias("s"),
      "s.customer_id = t.customer_id and s._commit_version = t._commit_version")
    .whenNotMatchedInsert(values = {"customer_id": "s.customer_id",
                              "company_name": "s.company_name",
                              "contact_email": "s.contact_email",
                              "address": "s.address",
                              "city": "s.city",
                              "postal_code": "s.postal_code",
                              "tier": "s.tier",
                              "industry": "s.industry",
                              "signup_date": "s.signup_date",
                              "start_at": "s.start_at",
                              "end_at": "s.end_at",
                              "is_current": "s.is_current",
                              "_insert_ts": "s._process_ts",
                              "_commit_version": "s._commit_version"})
    .execute()
    )

    # mark customers to be deleted as delete
    (target.alias("t").merge(
      refined_batch_df.filter(F.col("_change_type") == "delete").alias("s"),
      "s.customer_id = t.customer_id")
    .whenMatchedUpdate(set = {"is_current": F.lit(False),
                              "_is_delete": F.lit(True),
                              "_update_ts": "s._process_ts"})
    .execute()
    ) 

In [0]:
query = (df.writeStream
  .trigger(availableNow=True)
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .foreachBatch(merge_to_dim_customer)
  .start()
)

query.awaitTermination()